In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

In [ ]:
class CFG:
    model_name = "/kaggle/input/models/mohammedhamdan/mdeberta-v3-base/transformers/default/1/mdeberta-v3-base"

    # 按你的要求恢复原始长度配置
    max_len = 1024
    batch_size = 2
    valid_batch_size = 4
    grad_accum_steps = 4

    epochs = 5
    lr = 2e-5
    weight_decay = 0.01
    num_labels = 2
    valid_size = 0.2
    seed = 42

    num_workers = 2
    pin_memory = True
    use_amp = True
    max_grad_norm = 1.0

    best_model_path = "/kaggle/working/baseline_single_split.pth"
    resume_load_path = None   # 例如: "/kaggle/input/your-checkpoint-folder/baseline_single_split_last.pth"
    resume_save_path = "/kaggle/working/baseline_single_split_last.pth"
    save_last_checkpoint = True

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


In [ ]:
LABEL2ID = {
    "model_a": 0,
    "model_b": 1,
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [ ]:
def safe_to_text(x):
    if pd.isna(x):
        return ""
    return str(x)


def build_input_text(prompt, response_a, response_b):
    prompt = safe_to_text(prompt)
    response_a = safe_to_text(response_a)
    response_b = safe_to_text(response_b)

    text = (
        f"Prompt: {prompt}\n\n"
        f"Response A: {response_a}\n\n"
        f"Response B: {response_b}"
    )
    return text

In [ ]:
def load_data(train_path):
    df = pd.read_parquet(train_path).copy()

    # 构造输入文本
    df["input_text"] = df.apply(
        lambda x: build_input_text(x["prompt"], x["response_a"], x["response_b"]),
        axis=1
    )

    # 标签映射
    df["label"] = df["winner"].map(LABEL2ID)

    return df

In [ ]:
def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        CFG.model_name,
    )
    tokenizer.model_max_length = CFG.max_len
    return tokenizer

In [ ]:
class ArenaDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 按你的要求恢复为固定长度补齐
        encoded = self.tokenizer(
            row["input_text"],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
        )

        item = {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(row["label"], dtype=torch.long),
        }

        if "token_type_ids" in encoded:
            item["token_type_ids"] = torch.tensor(encoded["token_type_ids"], dtype=torch.long)

        return item


In [ ]:
def get_model():
    config = AutoConfig.from_pretrained(CFG.model_name)
    config.num_labels = CFG.num_labels

    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name,
        config=config
    )
    model = model.float()   # 强制转回 FP32
    return model


def prepare_model_for_multi_gpu(model):
    gpu_count = torch.cuda.device_count()

    model = model.to(CFG.device)

    if gpu_count > 1:
        print(f"Using DataParallel on {gpu_count} GPUs")
        model = nn.DataParallel(model)
    else:
        print(f"Using single device: {CFG.device}")

    return model


def get_model_to_save(model):
    return model.module if isinstance(model, nn.DataParallel) else model

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device, scaler):
    model.train()
    total_loss = 0.0

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, total=len(loader))):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=CFG.use_amp and device.type == "cuda"):
            if token_type_ids is not None:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels
                )
            else:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            loss = outputs.loss
            if loss.dim() > 0:
                loss = loss.mean()

        total_loss += loss.item()

        loss = loss / CFG.grad_accum_steps
        scaler.scale(loss).backward()

        should_step = ((step + 1) % CFG.grad_accum_steps == 0) or ((step + 1) == len(loader))
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    return total_loss / len(loader)

In [ ]:
@torch.no_grad()
def valid_one_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []

    for batch in tqdm(loader, total=len(loader)):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=CFG.use_amp and device.type == "cuda"):
            if token_type_ids is not None:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels
                )
            else:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

        loss = outputs.loss
        if loss.dim() > 0:
            loss = loss.mean()

        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)

        total_loss += loss.item()
        all_labels.extend(labels.detach().cpu().numpy())
        all_probs.extend(probs.detach().float().cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    preds = all_probs.argmax(axis=1)

    metrics = {
        "valid_loss": total_loss / len(loader),
        "log_loss": log_loss(all_labels, all_probs),
        "accuracy": accuracy_score(all_labels, preds),
        "macro_f1": f1_score(all_labels, preds, average="macro")
    }

    return metrics, all_probs, all_labels

In [ ]:
def run_single_split(train_path):
    set_seed(CFG.seed)

    print(f"device: {CFG.device}")
    print(f"gpu count: {torch.cuda.device_count()}")
    print(
        f"max_len={CFG.max_len}, batch_size={CFG.batch_size}, "
        f"valid_batch_size={CFG.valid_batch_size}, grad_accum_steps={CFG.grad_accum_steps}"
    )

    df = load_data(train_path)
    tokenizer = get_tokenizer()

    train_df, valid_df = train_test_split(
        df,
        test_size=CFG.valid_size,
        random_state=CFG.seed,
        stratify=df["label"]
    )

    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    print(f"train size: {len(train_df)}")
    print(f"valid size: {len(valid_df)}")

    train_dataset = ArenaDataset(train_df, tokenizer, CFG.max_len)
    valid_dataset = ArenaDataset(valid_df, tokenizer, CFG.max_len)

    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=CFG.pin_memory,
        persistent_workers=CFG.num_workers > 0,
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CFG.valid_batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=CFG.pin_memory,
        persistent_workers=CFG.num_workers > 0,
    )

    model = get_model()
    model = prepare_model_for_multi_gpu(model)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.lr,
        weight_decay=CFG.weight_decay
    )

    num_update_steps_per_epoch = int(np.ceil(len(train_loader) / CFG.grad_accum_steps))
    num_train_steps = num_update_steps_per_epoch * CFG.epochs

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_train_steps),
        num_training_steps=num_train_steps
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=CFG.use_amp and CFG.device.type == "cuda"
    )

    start_epoch = 0
    best_logloss = 1e9
    best_metrics = None

    resume_ckpt_path = CFG.resume_load_path
    if resume_ckpt_path is not None and os.path.exists(resume_ckpt_path):
        print(f"\nLoading resume checkpoint: {resume_ckpt_path}")
        checkpoint = torch.load(resume_ckpt_path, map_location=CFG.device)

        get_model_to_save(model).load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        if "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])

        start_epoch = checkpoint["epoch"] + 1
        best_logloss = checkpoint.get("best_logloss", best_logloss)
        best_metrics = checkpoint.get("best_metrics", best_metrics)

        print(f"Resumed from epoch {start_epoch}")
        print(f"best_logloss so far: {best_logloss:.4f}")

    for epoch in range(start_epoch, CFG.epochs):
        print(f"\n--- Epoch {epoch + 1} ---")

        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, CFG.device, scaler)
        metrics, _, _ = valid_one_epoch(model, valid_loader, CFG.device)

        print(f"train_loss: {train_loss:.4f}")
        print(f"valid_loss: {metrics['valid_loss']:.4f}")
        print(f"log_loss:   {metrics['log_loss']:.4f}")
        print(f"accuracy:   {metrics['accuracy']:.4f}")
        print(f"macro_f1:   {metrics['macro_f1']:.4f}")

        model_to_save = get_model_to_save(model)

        if metrics["log_loss"] < best_logloss:
            best_logloss = metrics["log_loss"]
            best_metrics = metrics.copy()
            torch.save(model_to_save.state_dict(), CFG.best_model_path)
            print(f"saved best model: {CFG.best_model_path}")

        if CFG.save_last_checkpoint:
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model_to_save.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "best_logloss": best_logloss,
                    "best_metrics": best_metrics,
                },
                CFG.resume_save_path,
            )
            print(f"saved resume checkpoint: {CFG.resume_save_path}")

    print("\n========== Best Result ==========")
    print(best_metrics)

    return best_metrics


In [ ]:
if __name__ == "__main__":
    train_path = "/kaggle/input/datasets/jingyeliu03/training-dataset/train.parquet"
    # 断点续训时，把 CFG.resume_load_path 设为你想加载的 checkpoint 完整路径；从头训练则设为 None
    run_single_split(train_path)
